In [1]:
import cobra
from cobra.util.array import create_stoichiometric_matrix
from cobra.flux_analysis import flux_variability_analysis, gapfill
import cobra.flux_analysis.gapfilling as gf

import gurobipy as gp 
import numpy as np
import pandas as pd

In [2]:
# Load the Human-GEM model
model = cobra.io.read_sbml_model("../data/Human-GEM.xml")
original_model = cobra.io.read_sbml_model("../data/Human-GEM.xml")

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-14


In [3]:
# Set solver parameters 

# Model 
model.solver = "gurobi"
model.solver.configuration.tolerances.integrality = 1e-9
model.solver.configuration.tolerances.optimality = 1e-9
model.solver.configuration.tolerances.feasibility = 1e-9

# Original model 
original_model.solver = "gurobi"
original_model.solver.configuration.tolerances.integrality = 1e-9
original_model.solver.configuration.tolerances.optimality = 1e-9
original_model.solver.configuration.tolerances.feasibility = 1e-9

In [4]:
# Load the medium file 
rpmi1640 = pd.read_csv("../../media/RPMI1640.tsv", sep="\t")
fbs = pd.read_csv("../../media/FBS.tsv", sep="\t")   
combined = pd.concat([rpmi1640, fbs], ignore_index=True)

group_cols = ['Mets_Recon3D', 'Mets_HumanGEM', 'ExRxns_Recon3D', 'ExRxns_HumanGEM']
medium = combined.groupby(group_cols, as_index=False)['Concentration_M'].sum()

# print(medium.head())

# Conc2Rate 
# def conc_to_flux(conc_M, cell_conc, cell_weight_g=400e-12, doubling_time_h=24):
#     return conc_M / (cell_conc * cell_weight_g * doubling_time_h * 1000)

# medium["Flux_mmol_gDW_h"] = conc_to_flux(medium["Concentration_M"], 1e5)


# Create a dictionary for medium fluxes
medium_fluxes = {
    row["ExRxns_HumanGEM"]: -1000.0
    for _, row in medium.iterrows()
}

mandatory_ex_rxns = {
    "MAR09048": -1000.0,  # O2
    "MAR09047": -1000.0,  # H2O
    "MAR09058": -1000.0,  # CO2
    "MAR09079": -1000.0,  # H+
}

medium_fluxes.update(mandatory_ex_rxns)

medium_fluxes

{'MAR08386': -1000.0,
 'MAR09363': -1000.0,
 'MAR09269': -1000.0,
 'MAR09066': -1000.0,
 'MAR09062': -1000.0,
 'MAR09070': -1000.0,
 'MAR09273': -1000.0,
 'MAR09109': -1000.0,
 'MAR09082': -1000.0,
 'MAR09083': -1000.0,
 'MAR09285': -1000.0,
 'MAR09150': -1000.0,
 'MAR09460': -1000.0,
 'MAR09293': -1000.0,
 'MAR09146': -1000.0,
 'MAR09034': -1000.0,
 'MAR09063': -1000.0,
 'MAR09071': -1000.0,
 'MAR09067': -1000.0,
 'MAR09351': -1000.0,
 'MAR09078': -1000.0,
 'MAR09038': -1000.0,
 'MAR09039': -1000.0,
 'MAR09361': -1000.0,
 'MAR09081': -1000.0,
 'MAR09040': -1000.0,
 'MAR09041': -1000.0,
 'MAR09042': -1000.0,
 'MAR09077': -1000.0,
 'MAR09378': -1000.0,
 'MAR09043': -1000.0,
 'MAR09072': -1000.0,
 'MAR09145': -1000.0,
 'MAR09393': -1000.0,
 'MAR09068': -1000.0,
 'MAR09396': -1000.0,
 'MAR09144': -1000.0,
 'MAR09147': -1000.0,
 'MAR09143': -1000.0,
 'MAR09069': -1000.0,
 'MAR09074': -1000.0,
 'MAR09159': -1000.0,
 'MAR09044': -1000.0,
 'MAR09045': -1000.0,
 'MAR09429': -1000.0,
 'MAR09064

In [12]:
print(model.reactions.get_by_id("MAR11985").name)
print(model.reactions.get_by_id("MAR11985").lower_bound)
print(model.reactions.get_by_id("MAR11985").upper_bound)

Exchange of High Density Lipoprotein
-1000.0
1000.0


In [9]:
# model.reactions.get_by_id("MAR11985").name

consistent_model = cobra.flux_analysis.fastcc(model)

ContainerAlreadyContains: Container '<optlang.container.Container object at 0x152cfc490>' already contains an object with name 'constraint_MAR03905'.

In [7]:
consistent_model.reactions.get_by_id("MAR11985").name   

KeyError: 'MAR11985'

In [6]:
S = create_stoichiometric_matrix(model)

exchange_rxns = []
for i, rxn in enumerate(model.reactions):
    if np.count_nonzero(S[:, i]) == 1:
        exchange_rxns.append(rxn)

print(len(exchange_rxns), "exchange reactions found")

# for rxn in exchange_rxns:
#     print(rxn.id, rxn.name)

1660 exchange reactions found


In [42]:
# Identify the transport reactions
# Get reactions involving transport between compartments

transport_rxns = []
transport_rxns_with_gpr = []
for rxn in model.reactions:
    compartments = set([met.compartment for met in rxn.metabolites])
    if len(compartments) > 1 and 'e' in compartments:
        print(rxn.id, rxn.name, "Compartments:", compartments)
        print(rxn)
        print(rxn.gene_reaction_rule)
        transport_rxns.append(rxn)
        if rxn.gene_reaction_rule:
            transport_rxns_with_gpr.append(rxn)
        print(rxn.annotation)

print(f"Number of transport reactions: {len(transport_rxns)}")
print(f"Number of transport reactions with GPR: {len(transport_rxns_with_gpr)}")

MAR08769  Compartments: {'e', 'c'}
MAR08769: MAM01307c + MAM02026c --> MAM00097e + MAM01626e
ENSG00000099998 or ENSG00000100031 or ENSG00000100121 or ENSG00000131067 or ENSG00000149435 or ENSG00000167741
{'sbo': 'SBO:0000176', 'ec-code': '2.3.2.2', 'bigg.reaction': 'GTMLTe', 'vmhreaction': 'GTMLTe', 'metanetx.reaction': 'MNXR100452'}
MAR08770  Compartments: {'e', 'c'}
MAR08770: MAM00097e --> MAM01127c + MAM01307c
ENSG00000006625 or ENSG00000134864
{'sbo': 'SBO:0000176', 'ec-code': '2.3.2.4', 'bigg.reaction': 'GGLUCT', 'vmhreaction': 'GGLUCT', 'metanetx.reaction': 'MNXR100126'}
MAR01617 Fatty Acid Transport via Diffusion Compartments: {'e', 'c'}
MAR01617: MAM03884e <=> MAM03884c

{'sbo': 'SBO:0000185', 'ec-code': '1.14.99.1', 'bigg.reaction': 'PHYTt', 'vmhreaction': 'PHYTt', 'metanetx.reaction': 'MNXR102660'}
MAR01852  Compartments: {'e', 'c'}
MAR01852: MAM01988c + MAM02357c <=> MAM01988e + MAM02357e
ENSG00000100652
{'sbo': 'SBO:0000185', 'vmhreaction': 'r0992', 'metanetx.reaction': 'MN

In [46]:
# Find complementary mets 
medium_mets = list((medium["Mets_HumanGEM"]))

rxns_with_medium_mets = []
for rxn in transport_rxns:
    for met in rxn.metabolites:
        if met.id in medium_mets:
            # print(rxn.id, rxn.name)
            rxns_with_medium_mets.append(rxn)

# Remove duplicates
rxns_with_medium_mets = list(set(rxns_with_medium_mets))

print(f"Number of transport reactions with medium metabolites: {len(rxns_with_medium_mets)}")

Number of transport reactions with medium metabolites: 1205


In [57]:
metabolites_not_in_med = []
for rxn in rxns_with_medium_mets: 
    # print(rxn.id, rxn.name)
    for met in rxn.metabolites:
        if met.id not in medium_mets and met.compartment == 'e':
            # print(rxn.id, rxn.name)
            metabolites_not_in_med.append(met.id)

# Remove duplicates 
metabolites_not_in_med = list(set(metabolites_not_in_med))

print(f"Number of metabolites not in medium: {len(metabolites_not_in_med)}")

Number of metabolites not in medium: 227


In [61]:
for met in metabolites_not_in_med:
    model_met = model.metabolites.get_by_id(met) 
    print(model_met.reactions)
    break 

frozenset({<Reaction MAR12523 at 0x3233cafd0>, <Reaction MAR12518 at 0x3233bb710>, <Reaction MAR12648 at 0x32346a490>, <Reaction MAR12522 at 0x3233cabd0>})


In [ ]:
other_ex_rxns = []
for met in metabolites_not_in_med:
    model_met = model.metabolites.get_by_id(met)
    for rxn in model_met.reactions:
        if rxn in exchange_rxns:
            other_ex_rxns.append(rxn.id)

other_ex_rxns = list(set(other_ex_rxns))
print(len(other_ex_rxns))

227
Other exchange reactions: ['MAR12675', 'MAR11949', 'MAR09921', 'MAR09418', 'MAR12705', 'MAR09093', 'MAR10243', 'MAR00621', 'MAR09386', 'MAR12586', 'MAR09281', 'MAR09259', 'MAR09923', 'MAR04151', 'MAR09271', 'MAR04234', 'MAR09690', 'MAR09412', 'MAR09129', 'MAR09065', 'MAR09290', 'MAR12713', 'MAR13068', 'MAR09091', 'MAR12221', 'MAR09292', 'MAR11891', 'MAR00619', 'MAR11426', 'MAR12618', 'MAR09137', 'MAR12204', 'MAR09073', 'MAR12232', 'MAR09148', 'MAR11379', 'MAR12648', 'MAR09424', 'MAR09296', 'MAR09362', 'MAR10192', 'MAR09809', 'MAR11390', 'MAR09297', 'MAR09139', 'MAR09169', 'MAR12607', 'MAR09920', 'MAR04226', 'MAR09140', 'MAR11454', 'MAR12716', 'MAR09092', 'MAR09367', 'MAR09142', 'MAR09086', 'MAR04148', 'MAR09167', 'MAR12216', 'MAR12697', 'MAR09165', 'MAR12653', 'MAR09407', 'MAR04238', 'MAR10205', 'MAR12227', 'MAR10081', 'MAR00573', 'MAR09295', 'MAR04203', 'MAR09808', 'MAR09702', 'MAR10191', 'MAR09844', 'MAR09395', 'MAR04213', 'MAR09364', 'MAR10188', 'MAR12702', 'MAR10345', 'MAR12228

In [69]:
# Find exchange reactions 
# ex_rxns = model.exchanges # Warning find only exchange (not demand or sink if Recon3D)

# Apply medium constraints to the model 
for rxn in exchange_rxns:
    reaction = model.reactions.get_by_id(rxn.id)
    if rxn.id in medium_fluxes:
        print(rxn.name)
        reaction.lower_bound = -1000.0
    elif rxn.id in other_ex_rxns:
        print(rxn.name)
        reaction.lower_bound = -1000.0
        # model.remove_reactions([reaction])
    else:
        reaction.lower_bound = 0.0
        # model.remove_reactions([reaction])

Exchange of propanoate
Exchange of butyrate
Exchange of glucose
Exchange of histidine
Exchange of isoleucine
Exchange of leucine
Exchange of lysine
Exchange of methionine
Exchange of phenylalanine
Exchange of threonine
Exchange of tryptophan
Exchange of valine
Exchange of H2O
Exchange of O2
Exchange of CO2
Exchange of alanine
Exchange of asparagine
Exchange of glutamine
Exchange of tyrosine
Exchange of cysteine
Exchange of arginine
Exchange of glycine
Exchange of proline
Exchange of serine
Exchange of aspartate
Exchange of glutamate
Exchange of Pi
Exchange of NH3
Exchange of sulfate
Exchange of urate
Exchange of Na+
Exchange of HCO3-
Exchange of H+
Exchange of K+
Exchange of Ca2+
Exchange of choline
Exchange of acetate
Exchange of ornithine
Exchange of sulfite
Exchange of 4-aminobutyrate
Exchange of dopamine
Exchange of noradrenaline
Exchange of D-aspartate
Exchange of D-alanine
Exchange of biotin
Exchange of estradiol-17beta 3-glucuronide
Exchange of pyruvate
Exchange of L-lactate
Exc

In [70]:
model.objective = 'MAR13082'
original_model.objective = 'MAR13082'  # Biomass reaction
print(model.optimize().objective_value)
print(original_model.optimize().objective_value)

0.0
124.86814837744572


In [10]:
solution = gapfill(model, original_model, demand_reactions=False)

# reactions = solution.fill()
# for reaction in reactions[0]:
#     print("Suggested:", reaction.id)

Read LP format model from file /var/folders/hd/x2nmhnb55t3_s_kjs_c8js6m0000gn/T/tmp2ygy5lmo.lp
Reading time = 0.04 seconds
: 8455 rows, 25942 columns, 110884 nonzeros
Read LP format model from file /var/folders/hd/x2nmhnb55t3_s_kjs_c8js6m0000gn/T/tmpxxny6d09.lp
Reading time = 0.04 seconds
: 8455 rows, 25942 columns, 110884 nonzeros


Infeasible: gap filling optimization failed (infeasible).

In [26]:
for i, entries in enumerate(solution.fill()):
    print("---- Run %d ----" % (i + 1))
    for e in entries:
        print(e.id)

RuntimeError: Failed to validate gap filled model, try lowering the integer threshold.

In [23]:
for reaction in reactions[0]:
    print("Suggested:", reaction.name)

Suggested: Exchange of VLDL remnant
Suggested: Exchange of heme
Suggested: Exchange of alpha-tocotrienol
Suggested: Exchange of lipoic acid
Suggested: Exchange of biomass


In [16]:
i = 0
for rxn in exchange_rxns:
    # print(rxn.id, rxn.lower_bound, rxn.upper_bound, rxn.name)
    if rxn.lower_bound < 0.0:
        print(rxn.id, rxn.lower_bound, rxn.upper_bound, rxn.name)
        # print(rxn.id, rxn.name)
        i += 1
print(i)

MAR09034 -1000.0 1000.0 Exchange of glucose
MAR09038 -1000.0 1000.0 Exchange of histidine
MAR09039 -1000.0 1000.0 Exchange of isoleucine
MAR09040 -1000.0 1000.0 Exchange of leucine
MAR09041 -1000.0 1000.0 Exchange of lysine
MAR09042 -1000.0 1000.0 Exchange of methionine
MAR09043 -1000.0 1000.0 Exchange of phenylalanine
MAR09044 -1000.0 1000.0 Exchange of threonine
MAR09045 -1000.0 1000.0 Exchange of tryptophan
MAR09046 -1000.0 1000.0 Exchange of valine
MAR09047 -1000.0 1000.0 Exchange of H2O
MAR09048 -1000.0 1000.0 Exchange of O2
MAR09058 -1000.0 1000.0 Exchange of CO2
MAR09062 -1000.0 1000.0 Exchange of asparagine
MAR09063 -1000.0 1000.0 Exchange of glutamine
MAR09064 -1000.0 1000.0 Exchange of tyrosine
MAR09066 -1000.0 1000.0 Exchange of arginine
MAR09067 -1000.0 1000.0 Exchange of glycine
MAR09068 -1000.0 1000.0 Exchange of proline
MAR09069 -1000.0 1000.0 Exchange of serine
MAR09070 -1000.0 1000.0 Exchange of aspartate
MAR09071 -1000.0 1000.0 Exchange of glutamate
MAR09072 -1000.0 1

In [15]:
fva_result = flux_variability_analysis(model, reaction_list=ex_rxns)

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-14
Read LP format model from file /var/folders/hd/x2nmhnb55t3_s_kjs_c8js6m0000gn/T/tmpyj5qyye5.lp
Reading time = 0.04 seconds
: 8456 rows, 25943 columns, 110887 nonzeros
Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-14
Read LP format model from file /var/folders/hd/x2nmhnb55t3_s_kjs_c8js6m0000gn/T/tmpzvx_v6yr.lp
Reading time = 0.04 seconds
: 8456 rows, 25943 columns, 110887 nonzeros
Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-14
Read LP format model from file /var/folders/hd/x2nmhnb55t3_s_kjs_c8js6m0000gn/T/tmpyarmo_js.lp
Reading time = 0.04 seconds
: 8456 rows, 25943 columns, 110887 nonzeros
Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-14
Read LP format model from file /var/folders/hd/x2nmhnb55t3_s_kjs_c8js6m0000gn/T/tmpb2mbk4yl.lp
Reading time = 0.06 seconds
: 8456 rows,

In [16]:
print(fva_result.head())

          minimum  maximum
MAR00659      0.0      0.0
MAR07108      0.0      0.0
MAR07110      0.0      0.0
MAR07112      0.0      0.0
MAR07114      0.0      0.0


In [17]:
# Check if any reactions have minimum > maximum
invalid_bounds = fva_result[fva_result['minimum'] > fva_result['maximum']]
if not invalid_bounds.empty:
    print("Reactions with minimum > maximum:")
    print(invalid_bounds)
else:
    print("No reactions have minimum > maximum.")

# Find reactions with minimum < 0
negative_min = fva_result[fva_result['minimum'] < 0]
print(f"\nNumber of reactions with minimum < 0: {len(negative_min)}")
print("List of reactions with minimum < 0:")
print(negative_min.index.tolist())


No reactions have minimum > maximum.

Number of reactions with minimum < 0: 46
List of reactions with minimum < 0:
['MAR09034', 'MAR09038', 'MAR09039', 'MAR09040', 'MAR09041', 'MAR09042', 'MAR09043', 'MAR09044', 'MAR09045', 'MAR09046', 'MAR09047', 'MAR09048', 'MAR09058', 'MAR09062', 'MAR09063', 'MAR09064', 'MAR09066', 'MAR09067', 'MAR09068', 'MAR09069', 'MAR09070', 'MAR09071', 'MAR09072', 'MAR09074', 'MAR09075', 'MAR09078', 'MAR09079', 'MAR09083', 'MAR09143', 'MAR09144', 'MAR09145', 'MAR09146', 'MAR09147', 'MAR09150', 'MAR09159', 'MAR09273', 'MAR09285', 'MAR09293', 'MAR09351', 'MAR09361', 'MAR09363', 'MAR09378', 'MAR09393', 'MAR09396', 'MAR09429', 'MAR08386']


In [18]:
# Assume `fva_result` is your DataFrame and `flux_dict` is your dictionary

# Step 1: Extract reactions
flux_dict_reactions = set(medium_fluxes.keys())
negative_min_reactions = set(fva_result[fva_result['minimum'] < 0].index)

# Step 2: Compare sets
common_reactions = flux_dict_reactions & negative_min_reactions
only_in_flux_dict = flux_dict_reactions - negative_min_reactions
only_in_fva_result = negative_min_reactions - flux_dict_reactions

# Step 3: Print results
print(f"Number of common reactions: {len(common_reactions)}")
print(f"Common reactions:\n{sorted(common_reactions)}")

print(f"\nReactions only in flux_dict: {len(only_in_flux_dict)}")
print(f"{sorted(only_in_flux_dict)}")

print(f"\nReactions only in FVA result with negative min: {len(only_in_fva_result)}")
print(f"{sorted(only_in_fva_result)}")

Number of common reactions: 46
Common reactions:
['MAR08386', 'MAR09034', 'MAR09038', 'MAR09039', 'MAR09040', 'MAR09041', 'MAR09042', 'MAR09043', 'MAR09044', 'MAR09045', 'MAR09046', 'MAR09047', 'MAR09048', 'MAR09058', 'MAR09062', 'MAR09063', 'MAR09064', 'MAR09066', 'MAR09067', 'MAR09068', 'MAR09069', 'MAR09070', 'MAR09071', 'MAR09072', 'MAR09074', 'MAR09075', 'MAR09078', 'MAR09079', 'MAR09083', 'MAR09143', 'MAR09144', 'MAR09145', 'MAR09146', 'MAR09147', 'MAR09150', 'MAR09159', 'MAR09273', 'MAR09285', 'MAR09293', 'MAR09351', 'MAR09361', 'MAR09363', 'MAR09378', 'MAR09393', 'MAR09396', 'MAR09429']

Reactions only in flux_dict: 7
['MAR09077', 'MAR09081', 'MAR09082', 'MAR09109', 'MAR09153', 'MAR09269', 'MAR09460']

Reactions only in FVA result with negative min: 0
[]
